# 02 — Aspect Extraction Evaluation Sample V2

## Objectives

Validate the corrected 240-review evaluation artifact for `hair_conditioners_extraction_eval_v2`. The audit proves identity alignment, discovery holdout, stratified-SRS population weights, unweighted targeted diagnostics, spreadsheet-safe annotation export, and the untouched 240-row pending queue.

## Цели

Проверить исправленный evaluation-артефакт из 240 отзывов для `hair_conditioners_extraction_eval_v2`. Аудит подтверждает согласованность identity, исключение discovery-выборки, корректные веса stratified SRS, отсутствие весов у целевой диагностики, безопасность CSV для таблиц и неизменённую очередь из 240 ожидающих строк.

In [ ]:
# Standard library and project discovery / Стандартная библиотека и поиск проекта
import hashlib
import json
import sys
from pathlib import Path

# Analysis and notebook display / Аналитика и отображение в ноутбуке
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reusable validation and CSV safety / Переиспользуемая валидация и безопасность CSV
from src.analytics.aspect_annotations import validate_aspect_annotation_queue
from src.common.csv_safety import escape_spreadsheet_formula
from src.common.project import find_project_root

In [ ]:
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
DATASET_VERSION = "amazon_reviews_2023_beauty_2021_2023_v1"
EVALUATION_VERSION = "hair_conditioners_extraction_eval_v2"
TAXONOMY_VERSION = "hair_conditioners_taxonomy_v1"

CONFIG_PATH = PROJECT_ROOT / "config/aspects/hair_conditioners_extraction_eval_v2.json"
TAXONOMY_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_taxonomy_v1.json"
REPORT_PATH = PROJECT_ROOT / "reports/aspects/hair_conditioners_extraction_eval_v2_sample_report.json"
SAMPLE_PATH = PROJECT_ROOT / "data/processed" / DATASET_VERSION / "aspect_evaluation" / EVALUATION_VERSION / "sample.parquet"
DISCOVERY_SAMPLE_PATH = PROJECT_ROOT / "data/processed" / DATASET_VERSION / "aspect_discovery/hair_conditioners_v2_sample.parquet"
QUEUE_PATH = PROJECT_ROOT / "notebooks/03_aspects/hair_conditioners_extraction_eval_v2_review_queue.csv"
GUIDE_PATH = PROJECT_ROOT / "docs/ASPECT_ANNOTATION_GUIDE.md"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 1. Validate identities, hashes, and holdout boundaries

The configuration, report, Parquet sample, approved taxonomy, discovery sample, and blind CSV queue must refer to one dataset, niche, taxonomy, evaluation version, and schema. The evaluation review IDs must be unique and disjoint from the corrected 1,500-review discovery sample.

## 1. Проверка identity, хешей и границ holdout

Конфигурация, отчёт, Parquet-выборка, утверждённая таксономия, discovery-выборка и слепая CSV-очередь должны относиться к одному dataset, niche, taxonomy, evaluation version и schema. Идентификаторы evaluation-отзывов должны быть уникальными и не пересекаться с исправленной discovery-выборкой из 1 500 отзывов.

In [ ]:
required_paths = [
    CONFIG_PATH,
    TAXONOMY_PATH,
    REPORT_PATH,
    SAMPLE_PATH,
    DISCOVERY_SAMPLE_PATH,
    QUEUE_PATH,
    GUIDE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing V2 evaluation inputs: {missing_paths}"

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
sample = pd.read_parquet(SAMPLE_PATH)
discovery_sample = pd.read_parquet(DISCOVERY_SAMPLE_PATH)
queue = pd.read_csv(QUEUE_PATH, keep_default_na=False)

identity_fields = [
    "evaluation_version",
    "evaluation_schema_version",
    "taxonomy_version",
    "niche_id",
    "niche_version",
    "dataset_version",
]
for field in identity_fields:
    assert config[field] == report[field]
    assert sample[field].nunique(dropna=False) == 1
    assert sample[field].iloc[0] == config[field]

assert config["evaluation_version"] == EVALUATION_VERSION
assert config["taxonomy_version"] == TAXONOMY_VERSION
assert config["dataset_version"] == DATASET_VERSION
assert taxonomy["status"] == "approved"
for field in ["taxonomy_version", "niche_id", "niche_version", "dataset_version"]:
    assert taxonomy[field] == config[field]
assert set(report["taxonomy_identity_fields_validated"]) == {
    "dataset_version", "niche_id", "niche_version", "taxonomy_version"
}
assert set(report["discovery_sample_identity_fields_validated"]) == {
    "dataset_version", "niche_id", "niche_version", "sample_schema_version"
}
assert discovery_sample["dataset_version"].nunique() == 1
assert discovery_sample["dataset_version"].iloc[0] == config["dataset_version"]
assert discovery_sample["niche_id"].nunique() == 1
assert discovery_sample["niche_id"].iloc[0] == config["niche_id"]
assert discovery_sample["niche_version"].nunique() == 1
assert discovery_sample["niche_version"].iloc[0] == config["niche_version"]
assert discovery_sample["sample_schema_version"].nunique() == 1
assert discovery_sample["sample_schema_version"].iloc[0] == config["discovery_sample_schema_version"]

assert len(sample) == len(queue) == report["total_review_count"] == 240
assert sample["review_id"].is_unique
assert queue["review_id"].is_unique
assert set(sample["review_id"]) == set(queue["review_id"])
assert set(sample["review_id"]).isdisjoint(set(discovery_sample["review_id"]))
assert report["discovery_overlap_count"] == 0
assert report["component_overlap_count"] == 0
assert report["discovery_sample_membership_validated"] is True
assert report["output_sha256"] == sha256_file(SAMPLE_PATH)
assert report["annotation_queue_sha256"] == sha256_file(QUEUE_PATH)

identity_summary = pd.Series(
    {
        "evaluation_version": config["evaluation_version"],
        "evaluation_schema_version": config["evaluation_schema_version"],
        "taxonomy_version": config["taxonomy_version"],
        "niche_version": config["niche_version"],
        "dataset_version": config["dataset_version"],
        "evaluation_reviews": len(sample),
        "discovery_reviews_excluded": len(discovery_sample),
        "discovery_overlap": report["discovery_overlap_count"],
        "sample_sha256_verified": True,
        "queue_sha256_verified": True,
    },
    name="value",
)
display(identity_summary.to_frame())

## 2. Prove the representative SRS weights

The 160 representative rows are sampled without replacement within rating × year × length strata. Every row carries the inverse review inclusion probability `N_h / n_h`; therefore the sum of its weights must recover the complete 112,525-review eligible holdout population. No per-product cap is part of this probability design.

## 2. Проверка весов репрезентативной SRS-части

160 репрезентативных строк отобраны без возвращения внутри слоёв rating × year × length. Вес каждой строки равен обратной вероятности включения отзыва `N_h / n_h`, поэтому сумма весов должна восстанавливать всю подходящую holdout-совокупность из 112 525 отзывов. Ограничение числа отзывов товара в вероятностном дизайне не применяется.

In [ ]:
representative = sample.loc[sample["sampling_component"] == "representative"].copy()
targeted = sample.loc[sample["sampling_component"] == "targeted"].copy()
assert len(representative) == config["representative_sample_size"] == report["representative_review_count"] == 160
assert len(targeted) == config["targeted_sample_size"] == report["targeted_review_count"] == 80
assert report["representative_sampling_design"] == "stratified_simple_random_sampling_without_replacement"
assert report["representative_product_cap_applied"] is False
assert representative["sampling_weight"].notna().all()
assert (representative["sampling_weight"] > 0).all()
assert representative["sampling_weight_semantics"].eq("inverse_review_inclusion_probability").all()

representative_population = report["representative_population_review_count"]
weight_total = representative["sampling_weight"].sum()
assert representative_population == report["eligible_holdout_review_count"] == 112525
assert abs(weight_total - representative_population) < 1e-6

stratum_audit = (
    representative.groupby("stratum_id", as_index=False)
    .agg(
        sample_rows=("review_id", "count"),
        population_rows=("population_stratum_count", "first"),
        sampling_weight=("sampling_weight", "first"),
        weighted_population=("sampling_weight", "sum"),
    )
)
assert len(stratum_audit) == len(report["representative_strata"])
assert (stratum_audit["sample_rows"] * stratum_audit["sampling_weight"] - stratum_audit["population_rows"]).abs().max() < 1e-6
assert abs(stratum_audit["weighted_population"].sum() - 112525) < 1e-6

component_summary = (
    sample.groupby("sampling_component")
    .agg(
        review_count=("review_id", "count"),
        product_count=("parent_asin", "nunique"),
        populated_weight_count=("sampling_weight", "count"),
    )
    .reset_index()
)
display(pd.Series({"representative_weight_sum": weight_total, "eligible_holdout_reviews": representative_population, "absolute_difference": abs(weight_total - representative_population)}, name="value").to_frame())
display(component_summary)
display(stratum_audit)

plot_data = stratum_audit.sort_values("weighted_population")
figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(plot_data["stratum_id"], plot_data["weighted_population"], color="#4C78A8")
axis.set_title("Representative population recovered by SRS weights")
axis.set_xlabel("Eligible holdout reviews represented")
axis.set_ylabel("Rating × year × length stratum")
axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
plt.show()

## 3. Keep targeted diagnostics unweighted

The 80 targeted rows are a deterministic alias-coverage, non-probability sample. They help diagnose rare aspects but must never be combined with representative rows for population prevalence or headline metrics. Their weights and stratum population counts must remain empty; the two-reviews-per-product cap applies only to targeted selection, not to the combined sample.

## 3. Целевая диагностика остаётся невзвешенной

80 целевых строк — детерминированная non-probability выборка по покрытию синонимов. Она помогает диагностировать редкие аспекты, но не должна объединяться с репрезентативными строками для оценки распространённости или основных метрик. Веса и размеры генеральных слоёв у неё должны оставаться пустыми; лимит два отзыва на товар относится только к целевому отбору, а не ко всей объединённой выборке.

In [ ]:
assert report["targeted_sampling_design"] == "deterministic_alias_coverage_nonprobability_sample"
assert report["targeted_sampling_weight_semantics"].startswith("none;")
assert targeted["sampling_weight"].isna().all()
assert targeted["population_stratum_count"].isna().all()
assert targeted["sample_stratum_count"].isna().all()
assert targeted["sampling_weight_semantics"].eq("not_applicable_targeted_nonprobability_sample").all()
assert targeted.groupby("parent_asin")["review_id"].size().max() <= config["maximum_reviews_per_product"] == report["targeted_maximum_reviews_per_product"] == 2
assert sample.groupby("parent_asin")["review_id"].size().max() == report["maximum_reviews_per_product"] == 4

target_coverage = pd.DataFrame(report["targeted_aspect_coverage"])
assert len(target_coverage) == taxonomy["aspect_count"] == 33
assert target_coverage["selected_target_review_count"].min() >= config["targeted_minimum_per_aspect"]
display(
    target_coverage.sort_values(
        ["selected_target_review_count", "aspect_id"]
    ).reset_index(drop=True)
)

## 4. Validate the blind, spreadsheet-safe annotation queue

All 240 rows must remain pending with empty human-label fields. Target aspect hints stay only in Parquet and are absent from the CSV. The validator enforces exact evidence rules once rows are completed. Formula safety is checked both on the saved queue and against malicious sentinel strings that begin with spreadsheet execution prefixes.

## 4. Проверка слепой и безопасной для таблиц очереди

Все 240 строк должны оставаться в статусе pending с пустыми полями ручной разметки. Подсказки целевых аспектов хранятся только в Parquet и отсутствуют в CSV. После заполнения валидатор потребует точные фразы-доказательства. Безопасность формул проверяется как для сохранённой очереди, так и на тестовых вредоносных строках с префиксами выполнения электронных таблиц.

In [ ]:
validation = validate_aspect_annotation_queue(QUEUE_PATH, TAXONOMY_PATH)
assert validation.review_count == 240
assert validation.completed_review_count == 0
assert validation.pending_review_count == 240
assert queue["annotation_status"].eq("pending").all()
assert queue[["human_aspect_ids", "human_evidence_json", "no_supported_aspect", "annotator_notes"]].eq("").all().all()
assert "target_aspect_ids" not in queue.columns

formula_sentinels = ["=1+1", "+1+1", "-1+1", "@SUM(A1:A2)", "  =1+1", "\t=1+1", "\r+1+1"]
escaped_sentinels = [escape_spreadsheet_formula(value) for value in formula_sentinels]
assert all(value.startswith("'") for value in escaped_sentinels)
assert escape_spreadsheet_formula("ordinary text") == "ordinary text"

def is_unsafe_spreadsheet_text(value: object) -> bool:
    if not isinstance(value, str) or not value:
        return False
    stripped = value.lstrip(" \t\r\n")
    return stripped.startswith(("=", "+", "-", "@")) or value[0] in "\t\r"

text_columns = queue.select_dtypes(include=["object", "string"]).columns
unsafe_counts = {
    column: int(queue[column].map(is_unsafe_spreadsheet_text).sum())
    for column in text_columns
}
assert sum(unsafe_counts.values()) == 0

safety_summary = pd.Series(
    {
        "queue_rows": validation.review_count,
        "pending_rows": validation.pending_review_count,
        "completed_rows": validation.completed_review_count,
        "hidden_target_hint_column_absent": True,
        "unsafe_saved_cells": sum(unsafe_counts.values()),
        "formula_sentinel_cases_escaped": len(escaped_sentinels),
    },
    name="value",
)
display(safety_summary.to_frame())
display(queue[["annotation_order", "sampling_component", "rating", "review_year", "review_text", "annotation_status"]].head(10))

## Conclusion

Evaluation V2 passes all checks: 160 representative rows recover exactly 112,525 eligible holdout reviews through valid stratified-SRS weights, while 80 targeted rows remain explicitly unweighted diagnostics. The 240 unique reviews cover 208 products, have no discovery overlap, and all annotation rows remain pending. The CSV exposes no target hints and passes spreadsheet-formula safety checks; no human labels were invented.

## Вывод

Evaluation V2 прошла все проверки: 160 репрезентативных строк корректными весами stratified SRS точно восстанавливают 112 525 подходящих holdout-отзывов, а 80 целевых строк явно остаются невзвешенной диагностикой. 240 уникальных отзывов охватывают 208 товаров, не пересекаются с discovery-выборкой, и все строки разметки имеют статус pending. CSV не содержит целевых подсказок и проходит проверку безопасности формул; ручные метки не выдумывались.